Uus tabel, kus on:

pat_id (patterns.id)

head_id (transaction.head_id)

transaction_id (transaction.id)

phrase_nr (pattern.phrase_nr)

root (transaction.lemma)

Uue tabeli loomiseks:

1. transaction_head.verb matchib patterns.verb
2. transaction_head.id matchib transactions.head_id  
3. patterns.deprel + kääne peab matchima transaction.deptrel+kääne 


In [1]:
import sqlite3
import pandas as pd

In [2]:
# verbimustrite andmebaas
#pattern_db = "../example_data/verb_patterns_actors.db"

# transaktsioonide andmebaas
transaction_db = "../example_data/v33_subset.db"

# Siia salvestuvad loodavad tabelid
vp_data_db = "../example_data/vp_data_actors.db"

transactions_table = "transaction_v2" # "trans.'transaction'"

# state of the verb (isikumäärus)
# alati: alati isikumäärus etc
STAT = 'alati' #'mitte_kunagi'

In [3]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()

In [4]:
# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{transaction_db}" AS trans')

### Luua uus tabel pat_tr_head, kus on:

head_id, pat_id, verb_word, phrase_nr, phrase_case, deprel (verbi deprel)

Tabeli loomine:

Teha join transaction_head tabeliga verbi alusel.


In [6]:
%%time

cur.execute("""
DROP TABLE IF EXISTS pat_tr_head_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE pat_tr_head_{stat} AS
SELECT DISTINCT
    head.id as head_id,
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.verb_compound as verb_compound,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.pat_deprel as pat_deprel
FROM 
    patterns_actors_{stat2} as pat
INNER JOIN 
    trans.transaction_head as head
ON
    pat.verb_word = head.verb

""".format(stat=STAT, stat2=STAT))

CPU times: user 3.62 ms, sys: 348 µs, total: 3.96 ms
Wall time: 7.8 ms


### Luua uus tabel patterns_transaction_actors_*, kus on:

head_id, pat_id, transaction_id, phrase_nr, verb_word, root_word, verb_deprel, word_deprel, pos, phrase_case

Join toimub pat_tr_head_v1 ja transaction vahel. 

Praegu on joini aluseks head_id, deprel ja feats. 

` NB!!! patterns tabeli kääne on abl/all/ad jne ja neid tulebks matchida feats veerus olevaga`

In [7]:
%%time 

cur.execute("""
DROP TABLE IF EXISTS patterns_transaction_actors_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE patterns_transaction_actors_{stat} AS
SELECT DISTINCT
    tbl1.head_id as head_id,
    tbl1.pat_id as pat_id,
    tr.id as transaction_id,
    tbl1.phrase_nr as phrase_nr,
    tbl1.verb_word as verb_word,
    tbl1.verb_compound as verb_compound,
    tr.lemma as root_word,
    tbl1.pat_deprel as pat_deprel,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tbl1.phrase_case as phrase_case,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus
    
FROM 
    pat_tr_head_{stat2} as tbl1
JOIN 
    {trans_tbl} as tr
ON 
    tbl1.head_id = tr.head_id
    and tbl1.pat_deprel = tr.deprel
WHERE INSTR(',' || tr.feats || ',', ',' || tbl1.phrase_case || ',') > 0

""".format(stat=STAT, stat2=STAT, trans_tbl=transactions_table))

CPU times: user 4.73 ms, sys: 1.34 ms, total: 6.07 ms
Wall time: 12 ms


### kontroll

In [8]:
query = """SELECT * from patterns_transaction_actors_{stat}""".format(stat=STAT)
source = pd.read_sql_query(query, con)
source

,head_id,pat_id,transaction_id,phrase_nr,verb_word,verb_compound,root_word,pat_deprel,word_deprel,pos,phrase_case,tr_feats,koht,elus
0,179,2,258,1,nõudma,,mina,obl,obl,P,abl,"abl,sg",UNK,YES
1,10,120,19,1,tulema,,sina,obl,obl,P,ad,"ad,sg",UNK,YES
2,86,120,132,1,tulema,,tema,obl,obl,P,ad,"ad,pl",UNK,UNK
3,92,121,138,1,tekkima,,mina,obl,obl,P,ad,"ad,sg",UNK,YES
4,189,121,277,1,tekkima,,mina,obl,obl,P,ad,"ad,sg",UNK,YES
5,56,151,92,1,meeldima,,mina,obl,obl,P,all,"all,sg",UNK,YES
6,193,151,285,1,meeldima,,mina,obl,obl,P,all,"all,sg",UNK,YES
7,264,154,401,1,helistama,,mina,obl,obl,P,all,"all,sg",UNK,YES
8,166,167,237,1,tundma,kaasa,tema,obl,obl,P,all,"all,pl",UNK,UNK
9,33,185,59,1,minema,peale,rahvas,obl,obl,S,all,"all,com,sg",UNK,UNK


In [9]:
con.close()